# ConSwinTX-Lite — 5-Fold Cross-Validation (PyTorch)

**Model Architecture (ConSwinTX-Lite):**
- CNN Branch  : MobileNetV2 `features[:8]` → [B, 64, 14, 14]
- Swin Branch : Custom Swin-T (embed_dim=8, depths=[2,2,2,2], patch_size=14, window_size=4)
  → 1×1 proj (64→48) → Upsample (2×2 → 14×14)
- Fusion      : concat [B, 112, 14, 14] → InvertedResBlock → [B, 128, 7, 7] → GAP → Linear(38)
- `NUM_WORKERS = 0` for Windows/Local Jupyter stability
- Full fault-tolerance and early stopping included.

In [ ]:
# ── 0. Install Libraries ──
import subprocess, sys
pkgs = ['codecarbon', 'torchvision', 'timm', 'fvcore', 'torchinfo']
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], check=False)
print('Dependencies ready.')

In [ ]:
# ── 1. Imports & Configuration ──
import os, glob, math, time, random, json, warnings, csv
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings('ignore')

SEED        = 42
IMAGE_SIZE  = 224
NUM_CLASSES = 38
BATCH_SIZE  = 128
EPOCHS      = 20
N_FOLDS     = 5
NUM_WORKERS = 0  # Optimal for Local Windows PC to prevent BrokenPipe

# ConSwinTX-Lite Model Config
PATCH_SIZE  = 14
EMBED_DIM   = 8
WINDOW_SIZE = 4
NUM_HEADS   = [1, 2, 4, 8]
DEPTHS      = [2, 2, 2, 2]
MLP_RATIO   = 2.0
DROP_RATE   = 0.1

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_ROOT = './conswintx_lite_cv'
os.makedirs(OUTPUT_ROOT, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print(f'Device: {DEVICE}')

In [ ]:
# ── 2. Download Dataset ──
import urllib.request, zipfile
ZIP_URL = 'https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip'
ZIP_FILE = 'master.zip'
if not os.path.exists(ZIP_FILE):
    print('Downloading dataset...')
    urllib.request.urlretrieve(ZIP_URL, ZIP_FILE)
if not os.path.exists('PlantVillage-Dataset-master'):
    print('Unzipping...')
    with zipfile.ZipFile(ZIP_FILE, 'r') as z: z.extractall('.')
print('Dataset Ready!')

In [ ]:
# ── 3. Helper Modules & Custom Layers ──

def _make_divisible(v, divisor, min_value=None):
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


class SEBlock(nn.Module):
    """Squeeze-and-Excitation (SE) block."""
    def __init__(self, in_channels, ratio=4):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // ratio, 1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // ratio, in_channels, 1, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.fc(self.avg_pool(x))


def window_partition(x, window_size):
    """Partitions feature map into non-overlapping windows."""
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    """Reverses the window partition back to the original feature map layout."""
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


def compute_mask(H, W, window_size, shift_size, device):
    """Generates the attention mask for cyclic shifted window attention."""
    img_mask = torch.zeros((1, H, W, 1), device=device)
    h_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))
    w_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))
    cnt = 0
    for h in h_slices:
        for w in w_slices:
            img_mask[:, h, w, :] = cnt
            cnt += 1
    mask_windows = window_partition(img_mask, window_size)
    mask_windows = mask_windows.view(-1, window_size * window_size)
    attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
    attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
    return attn_mask


class WindowAttention(nn.Module):
    """Window-based Multi-head Self Attention (W-MSA / SW-MSA) with relative position bias."""
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.window_size = window_size  # (Wh, Ww)
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads)
        )

        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing='ij'))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += self.window_size[0] - 1
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer('relative_position_index', relative_position_index)

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))

        rel_pos_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1
        )
        rel_pos_bias = rel_pos_bias.permute(2, 0, 1).contiguous()
        attn = attn + rel_pos_bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
            attn = F.softmax(attn, dim=-1)
        else:
            attn = F.softmax(attn, dim=-1)

        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class SwinTransformerBlock(nn.Module):
    """Standard / Shifted Window Swin Transformer Block."""
    def __init__(self, dim, num_heads, window_size=7, shift_size=0,
                 mlp_ratio=2., qkv_bias=True, dropout=0.1):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim, window_size=(window_size, window_size), num_heads=num_heads,
            qkv_bias=qkv_bias, attn_drop=dropout, proj_drop=dropout
        )

        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(dim * mlp_ratio), dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        H, W = self.H, self.W
        B, L, C = x.shape
        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)

        # Only apply cyclic shift if spatial dims are strictly larger than window size
        if self.shift_size > 0 and H > self.window_size and W > self.window_size:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
            attn_mask = compute_mask(H, W, self.window_size, self.shift_size, x.device)
        else:
            shifted_x = x
            attn_mask = None

        x_windows = window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        attn_windows = self.attn(x_windows, mask=attn_mask)
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_windows, self.window_size, H, W)

        if self.shift_size > 0 and H > self.window_size and W > self.window_size:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x

        x = x.view(B, H * W, C)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x


class PatchMerging(nn.Module):
    """Downsamples resolution by 2x and doubles feature depth."""
    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = nn.LayerNorm(4 * dim)

    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        x = x.view(B, H, W, C)
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], -1).view(B, -1, 4 * C)
        x = self.norm(x)
        return self.reduction(x)


class PatchEmbed(nn.Module):
    """Converts image into patch token sequence."""
    def __init__(self, img_size=224, patch_size=14, in_chans=3, embed_dim=8):
        super().__init__()
        self.patches_resolution = [img_size // patch_size, img_size // patch_size]
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        return self.norm(x)


class SwinStage(nn.Module):
    """Alternating W-MSA and SW-MSA Swin blocks, with optional downsampling."""
    def __init__(self, dim, depth, num_heads, window_size, input_resolution,
                 mlp_ratio=2., downsample=None):
        super().__init__()
        # Clamp window_size to input resolution so it always divides evenly
        effective_ws = min(window_size, input_resolution[0], input_resolution[1])
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                dim=dim, num_heads=num_heads, window_size=effective_ws,
                shift_size=0 if (i % 2 == 0) else effective_ws // 2,
                mlp_ratio=mlp_ratio
            ) for i in range(depth)
        ])
        self.downsample = downsample(input_resolution, dim=dim) if downsample is not None else None

    def forward(self, x, H, W):
        for block in self.blocks:
            block.H, block.W = H, W
            x = block(x)
        if self.downsample is not None:
            x = self.downsample(x)
            H, W = H // 2, W // 2
        return x, H, W


class CustomSwinTBranch(nn.Module):
    """
    Lightweight Swin-T variant.
    embed_dim=8, depths=[2,2,2,2], num_heads=[1,2,4,8], mlp_ratio=2
    patch_size=14  →  resolution 16x16

    Stage dims  : 8 → 16 → 32 → 64
    Resolutions : 16x16 → 8x8 → 4x4 → 2x2
    Final output channels: embed_dim * 2^3 = 64
    """
    def __init__(self, img_size=IMAGE_SIZE, patch_size=PATCH_SIZE, in_chans=3,
                 embed_dim=EMBED_DIM, depths=None, num_heads=None,
                 window_size=WINDOW_SIZE, mlp_ratio=MLP_RATIO):
        super().__init__()
        if depths is None:    depths    = DEPTHS
        if num_heads is None: num_heads = NUM_HEADS
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size,
            in_chans=in_chans, embed_dim=embed_dim
        )
        H, W = self.patch_embed.patches_resolution
        self.patches_resolution = [H, W]

        self.stages = nn.ModuleList()
        for i in range(len(depths)):
            stage_res = (H // (2 ** i), W // (2 ** i))
            stage_dim = embed_dim * (2 ** i)
            downsample = PatchMerging if (i < len(depths) - 1) else None
            self.stages.append(SwinStage(
                dim=stage_dim, depth=depths[i], num_heads=num_heads[i],
                window_size=window_size, input_resolution=stage_res,
                mlp_ratio=mlp_ratio, downsample=downsample
            ))

        self.final_dim = embed_dim * (2 ** (len(depths) - 1))  # 64
        self.norm = nn.LayerNorm(self.final_dim)

    def forward(self, x):
        x = self.patch_embed(x)
        H, W = self.patches_resolution
        for stage in self.stages:
            x, H, W = stage(x, H, W)
        x = self.norm(x)
        B, _, C = x.shape
        return x.view(B, H, W, C).permute(0, 3, 1, 2).contiguous()


class InvertedResBlock(nn.Module):
    def __init__(self, in_channels, expansion, stride, alpha, filters):
        super(InvertedResBlock, self).__init__()
        self.stride = stride
        pointwise_conv_filters = int(filters * alpha)
        self.pointwise_filters = _make_divisible(pointwise_conv_filters, 8)
        hidden_dim = expansion * in_channels
        self.use_res_connect = self.stride == 1 and in_channels == self.pointwise_filters

        layers = []
        if expansion != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, kernel_size=1, bias=False),
                nn.BatchNorm2d(hidden_dim, eps=1e-3, momentum=0.001),
                nn.ReLU6(inplace=True)
            ])
        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, stride=stride,
                      padding=1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim, eps=1e-3, momentum=0.001),
            nn.ReLU6(inplace=True)
        ])
        layers.extend([
            nn.Conv2d(hidden_dim, self.pointwise_filters, kernel_size=1, bias=False),
            nn.BatchNorm2d(self.pointwise_filters, eps=1e-3, momentum=0.001)
        ])
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_res_connect:
            return x + self.conv(x)
        return self.conv(x)

In [ ]:
# ── 4. The ConSwinTX-Lite Model ──
class HybridMobileNetSwin(nn.Module):
    """
    ConSwinTX-Lite (Ultra-Efficient Configuration)

    CNN branch  : MobileNetV2 features[:8]  -> [B,  64, 14, 14]
    Swin branch : CustomSwinTBranch (8-dim) -> [B,  64,  2,  2]
                  1x1 projection (64->48)   -> [B,  48,  2,  2]
                  Upsampled to (14, 14)     -> [B,  48, 14, 14]
    Fusion      : concat -> [B, 112, 14, 14] -> InvertedResBlock -> [B, 128, 7, 7]
    Output      : GAP -> Dropout -> Linear(128, num_classes)

    fusion_channels = 64 + 48 = 112
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super(HybridMobileNetSwin, self).__init__()

        # --- CNN Branch ---
        mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        self.cnn_base = nn.Sequential(*list(mobilenet.features.children())[:8])  # [B, 64, 14, 14]
        self.cnn_se = SEBlock(in_channels=64, ratio=4)

        # --- Swin Branch (embed_dim=8 -> final 64 channels) ---
        self.swin_branch = CustomSwinTBranch()
        # 1x1 projection: reduce 64 -> 48 channels
        self.swin_proj = nn.Conv2d(64, 48, kernel_size=1, bias=False)
        # Upsample 2x2 -> 14x14 to align with CNN branch
        self.swin_upsample = nn.Upsample(size=(14, 14), mode='bilinear', align_corners=False)
        self.swin_se = SEBlock(in_channels=48, ratio=4)

        # --- Fusion & Classification ---
        # fusion_channels = 64 (CNN) + 48 (Swin projected) = 112
        fusion_channels = 64 + 48
        self.fusion_block = InvertedResBlock(
            in_channels=fusion_channels, expansion=6, stride=2, alpha=1.0, filters=128
        )
        self.fusion_se = SEBlock(in_channels=128, ratio=4)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(DROP_RATE)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        # 1. CNN branch [B, 64, 14, 14]
        cnn_feat = self.cnn_se(self.cnn_base(x))

        # 2. Swin branch -> project -> upsample [B, 48, 14, 14]
        swin_feat = self.swin_branch(x)
        swin_feat = self.swin_proj(swin_feat)
        swin_feat = self.swin_upsample(swin_feat)
        swin_feat = self.swin_se(swin_feat)

        # 3. Concat [B, 112, 14, 14]
        fused = torch.cat([cnn_feat, swin_feat], dim=1)

        # 4. Fuse and classify
        out = self.fusion_se(self.fusion_block(fused))
        out = self.gap(out).flatten(1)
        return self.classifier(self.dropout(out))


_m = HybridMobileNetSwin(num_classes=NUM_CLASSES).to(DEVICE)
print('=' * 55)
print(f'  Model: ConSwinTX-Lite')
print(f'  Total Parameters: {sum(p.numel() for p in _m.parameters()):>12,}')
print('=' * 55)
del _m; torch.cuda.empty_cache()

In [ ]:
# ── 5. Data Processing & DataLoaders ──
DATASET_DIR = './PlantVillage-Dataset-master/raw/color'
classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
all_paths, all_labels = [], []
for label_idx, cls_name in enumerate(classes):
    imgs = sorted(glob.glob(os.path.join(DATASET_DIR, cls_name, '*.*')))
    all_paths.extend(imgs); all_labels.extend([label_idx] * len(imgs))
all_paths, all_labels = np.array(all_paths), np.array(all_labels)

pool_paths, test_paths, pool_labels, test_labels = train_test_split(
    all_paths, all_labels, test_size=0.10, random_state=SEED, stratify=all_labels)

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tfm = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
val_tfm = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class PVDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths, self.labels, self.transform = paths, labels, transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img) if self.transform else img, int(self.labels[idx])

def make_loader(paths, labels, training=False, batch_size=BATCH_SIZE):
    return DataLoader(PVDataset(paths, labels, train_tfm if training else val_tfm), batch_size=batch_size, shuffle=training, num_workers=NUM_WORKERS, pin_memory=True)

test_loader = make_loader(test_paths, test_labels, training=False)

In [ ]:
# ── 6. Train & Eval Loops ──
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train(); running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        running_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item(); total += imgs.size(0)
    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval(); running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast(): logits = model(imgs); loss = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item(); total += imgs.size(0)
    return running_loss / total, correct / total

def plot_training_curves(history, fold_dir, fold_num):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history['train_acc'], label='Train'); axes[0].plot(history['val_acc'], label='Val')
    axes[0].set_title('Accuracy'); axes[0].legend()
    axes[1].plot(history['train_loss'], label='Train'); axes[1].plot(history['val_loss'], label='Val')
    axes[1].set_title('Loss'); axes[1].legend()
    plt.savefig(os.path.join(fold_dir, 'training_curves.png'))
    plt.close()

In [ ]:
# ── 7. The 5-Fold Cross-Validation Loop ──
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_summaries = []

for fold_num, (train_idx, val_idx) in enumerate(skf.split(pool_paths, pool_labels), start=1):
    fold_dir = os.path.join(OUTPUT_ROOT, f'fold_{fold_num}')
    os.makedirs(fold_dir, exist_ok=True)
    json_path = os.path.join(fold_dir, 'fold_summary.json')

    if os.path.exists(json_path):
        print(f'\n[\u2713] FOLD {fold_num} Already Completed. Skipping.')
        with open(json_path, 'r') as f: fold_summaries.append(json.load(f))
        continue

    print(f'\n[>] STARTING FOLD {fold_num}/{N_FOLDS}')
    train_loader = make_loader(pool_paths[train_idx], pool_labels[train_idx], training=True)
    val_loader = make_loader(pool_paths[val_idx], pool_labels[val_idx], training=False)

    model = HybridMobileNetSwin(num_classes=NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)
    scaler = GradScaler()

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc, best_val_loss, patience_counter, PATIENCE, start_epoch = 0.0, float('inf'), 0, 5, 1
    ckpt_path = os.path.join(fold_dir, 'best_model.pt')
    latest_ckpt_path = os.path.join(fold_dir, 'latest_checkpoint.pt')

    if os.path.exists(latest_ckpt_path):
        checkpoint = torch.load(latest_ckpt_path, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state']); optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state']); scaler.load_state_dict(checkpoint['scaler_state'])
        history = checkpoint['history']; best_val_acc = checkpoint['best_val_acc']
        best_val_loss = checkpoint['best_val_loss']; patience_counter = checkpoint['patience_counter']
        start_epoch = checkpoint['epoch'] + 1
        print(f"  [!] Resuming from Epoch {start_epoch}")

    for epoch in range(start_epoch, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        scheduler.step(vl_loss)
        current_lr = optimizer.param_groups[0]['lr']

        for k, v in zip(['train_loss', 'train_acc', 'val_loss', 'val_acc'], [tr_loss, tr_acc, vl_loss, vl_acc]): history[k].append(v)

        print(f'  Epoch {epoch:>2}/{EPOCHS} | Train acc: {tr_acc*100:.2f}% | Val acc: {vl_acc*100:.2f}% | LR: {current_lr:.2e}')

        if vl_acc > best_val_acc:
            best_val_acc, best_val_loss, patience_counter = vl_acc, vl_loss, 0
            torch.save(model.state_dict(), ckpt_path)
            print(f'    \u2713 Best Checkpoint saved ({best_val_acc*100:.2f}%)')
        else: patience_counter += 1

        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'scheduler_state': scheduler.state_dict(), 'scaler_state': scaler.state_dict(), 'history': history, 'best_val_acc': best_val_acc, 'best_val_loss': best_val_loss, 'patience_counter': patience_counter}, latest_ckpt_path)

        if patience_counter >= PATIENCE:
            print(f'    [!] Early stopping at epoch {epoch}.'); break

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    plot_training_curves(history, fold_dir, fold_num)
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    print(f'\nFold {fold_num} Final Test Accuracy: {test_acc*100:.2f}%')

    fold_info = {'fold': fold_num, 'best_val_accuracy': best_val_acc, 'test_accuracy': test_acc}
    with open(json_path, 'w') as f: json.dump(fold_info, f)
    fold_summaries.append(fold_info)

    del model, train_loader, val_loader, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

print('\n[\u2713] ALL FOLDS COMPLETE')
val_accs  = [s['best_val_accuracy'] for s in fold_summaries]
test_accs = [s['test_accuracy'] for s in fold_summaries]
print(f'Mean Test Acc: {np.mean(test_accs)*100:.2f}% \u00b1 {np.std(test_accs)*100:.2f}%')